# RAG Evaluation challenge 0
### Develop an simple RAG application which gathers information from internet about the broad topics (user input) and uses the information as context for the LLM model and generate the output for the user query for same topic. Once the LLM genrates the response evaluate the response using the LLM and score it based on specified criterias.

## Planning
* Wide topic to be broken into smaller topics Ex: Topic : World's Leaders  -> List Important countries and their important leaders for past 100 years.
* Gether detailed information on the sub topics. Ex: Get the detailed information for about each leader on sertain topics like about life, education, family, struggles, contributions and awards and achievements.
*  Genretate test embaddings for the collected information and store them to vector store
* Genrate text embaddings for the user query and fetch the related documents from the vector store.
* Ingest the fetched docs and the user query to the LLM to generate the response.
* Evaluate the response generated by the LLM and score the response using LLM.
* if the confidence/score for the generated response is less then 60%, regenerate the response (maximum three ties)
* Reply to the user with the response and get the feedback.
* if the feed back is Positive or neutral, do nothing, else generate a new response considering the user feedback.



In [1]:
from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader, WikipediaLoader
from langchain_core.output_parsers import PydanticOutputParser
# from langchain_community.tools import DuckDuckGoSearchRun
# from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from typing import List, Literal, Annotated, Optional
from pydantic import Field, BaseModel, AnyUrl
# from langchain_core.runnables import RunnableSequence, RunnableBranch, RunnableLambda, RunnableParallel, RunnablePassthrough 
from dotenv import load_dotenv

/home/mysudarshan/Documents/AIML/NLP/envNLP/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_152478/3582961564.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader, WikipediaLoader, wikipedia
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# Loading the environement variables
load_dotenv()

True

In [3]:
# Describle Model output for a user topic
class Topic(BaseModel):
    regions : List[str] = Field(description="Sub-locations where topic has high presence/significance")
    names: dict[str, List[str]] = Field(description="Important list of names that works as examples for the topic region wise")
    details : List[str] =Field(description="Areas one should know about names")


In [4]:
# Prepare LLM Model
model_0 = ChatMistralAI(model_name='mistral-small-2603', temperature=0.4, timeout=60)
model_str_topic = model_0.with_structured_output(Topic) 

In [5]:
py_parser = PydanticOutputParser(pydantic_object= Topic )

In [6]:
sub_topic_promt = PromptTemplate(template='''You are a highly advance research bot who has deep understanding about {topic}. Break down the given topic   in to sub-topics based on the regions, also get the list of the details people should know about the sub-topics. Get the response in required output format. Generate output in format {format_instruction}''', 
                                 input_variables=['topic'],
                                 partial_variables={'format_instruction': py_parser.get_format_instructions()})

In [7]:
py_parser = PydanticOutputParser(pydantic_object= Topic )

In [8]:
topic = 'World Leaders from 20th century'

In [9]:
topic2 = 'Wars around the World in past 2 centuries'

In [10]:
sub_topics_chain_0 = sub_topic_promt | model_0

In [20]:
sub_topics_0 = sub_topics_chain_0.invoke({'topic' : topic2})

In [21]:
sub_topics_0.pretty_print()

================================== Ai Message ==================================

```json
{
  "regions": [
    "Europe",
    "Asia",
    "Africa",
    "Middle East",
    "North America",
    "South America",
    "Oceania"
  ],
  "names": {
    "Europe": [
      "Napoleonic Wars (1803–1815)",
      "World War I (1914–1918)",
      "World War II (1939–1945)",
      "Franco-Prussian War (1870–1871)",
      "Crimean War (1853–1856)"
    ],
    "Asia": [
      "Sino-Japanese Wars (1894–1895, 1937–1945)",
      "Russo-Japanese War (1904–1905)",
      "Korean War (1950–1953)",
      "Vietnam War (1955–1975)",
      "Indo-Pakistani Wars (1947, 1965, 1971, 1999)"
    ],
    "Africa": [
      "Anglo-Zulu War (1879)",
      "Second Boer War (1899–1902)",
      "Algerian War (1954–1962)",
      "Rwandan Civil War (1990–1994)",
      "Ethiopian-Eritrean War (1998–2000)"
    ],
    "Middle East": [
      "Arab-Israeli Conflict (1948–present)",
      "Iran-Iraq War (1980–1988)",
      "Gulf War (1990

In [22]:
sub_topics_chain_1 = sub_topic_promt | model_0 | py_parser

In [31]:
sub_topics_1 = sub_topics_chain_1.invoke({'topic' : topic2})

In [32]:
print(sub_topics_1.regions, sub_topics_1.names, sub_topics_1.details, sep='\n')

['Europe', 'Asia', 'Africa', 'Middle East', 'North America', 'South America', 'Oceania']
{'Europe': ['Napoleonic Wars', 'World War I', 'World War II', 'Franco-Prussian War', 'Crimean War'], 'Asia': ['Sino-Japanese War', 'Russo-Japanese War', 'Korean War', 'Vietnam War', 'Indo-Pakistani Wars'], 'Africa': ['Second Boer War', 'First Italo-Ethiopian War', 'Rwandan Civil War', 'Algerian War', 'Angolan Civil War'], 'Middle East': ['Iran-Iraq War', 'Six-Day War', 'Yom Kippur War', 'Gulf War', 'Syrian Civil War'], 'North America': ['American Civil War', 'Mexican-American War', 'War of 1812', 'Spanish-American War', 'Gulf War'], 'South America': ['Chaco War', 'War of the Pacific', 'Falklands War', 'Colombian Civil War', 'Paraguayan War'], 'Oceania': ['New Zealand Wars', 'Pacific Theater (World War II)', 'Fijian Coups', 'Bougainville Civil War', 'Solomon Islands Civil War']}
['Major conflicts and their causes', 'Key battles and turning points', 'Impact on geopolitics and borders', 'Significant t

## Step 2. Webscapping about the sub-topics(names)

In [65]:
import asyncio, time
async def wikipediaScapper(keyword : str, max_count=2 , dir=None ) -> List:
    '''Search wikipedia based on given keyword and store the articles to the given path.
        Input Parameters 
            keyword : Topic for the search
            max_count (Default=2) : Maximum document count 
            dir (Optional) : Directory to save data as text File 
        Output 
            str: Content for the keyword or file path 
    '''
    print(f'Looking Wikipedia for {keyword}')
    doc=None
    splitter = RecursiveCharacterTextSplitter(separators=['\n\n', '\n'], chunk_size = 200, chunk_overlap = 20)
    tries = 3
    SEM = asyncio.Semaphore(3)
    while tries:
        try:
            # wikiLoader = WikipediaLoader(query= keyword, load_max_docs=max_count)
            # doc = await asyncio.to_thread(wikiLoader.load)
            # tries=0
            async with SEM:
                wikiLoader = WikipediaLoader(query=keyword, load_max_docs=max_count)
                doc = await asyncio.to_thread(wikiLoader.load)
            tries = 0
        except Exception as error:
            tries-=1
            await asyncio.sleep(1)
            if not tries:
                print(error)
    print(f'Finished Task about {keyword}...')
    return doc


async def getherDataOnTopics(topics: List[str]):
    '''Asyncronously gets the data from internet for the topics provided.
        input : 
            topics : list of topics to gether information for. 
    '''
    tasks = [wikipediaScapper(topic, 1) for topic in topics ]
    result = await asyncio.gather(*tasks, return_exceptions=True)
    # docs = [d for d in result if d != None]
    docs=[]
    for doc in result:
        if doc:
            docs.append(*doc)

    print('All topics processed...', f'Total Docs count {len(docs)}.' ) 
    return docs

sub_topics = set()
for regions, sub_topic in sub_topics_1.names.items(): 
    sub_topics = sub_topics.union(set(sub_topic[:2]))
sub_topics = list(sub_topics)
print(len(sub_topic))
docs = await getherDataOnTopics(topics=sub_topics)

docs

5
Looking Wikipedia for Second Boer War
Looking Wikipedia for Napoleonic Wars
Looking Wikipedia for American Civil War
Looking Wikipedia for Iran-Iraq War
Looking Wikipedia for World War I
Looking Wikipedia for Sino-Japanese War
Looking Wikipedia for Russo-Japanese War
Looking Wikipedia for First Italo-Ethiopian War
Looking Wikipedia for New Zealand Wars
Looking Wikipedia for War of the Pacific
Looking Wikipedia for Six-Day War
Looking Wikipedia for Pacific Theater (World War II)
Looking Wikipedia for Chaco War
Looking Wikipedia for Mexican-American War
Finished Task about Chaco War...
Finished Task about Napoleonic Wars...
Finished Task about War of the Pacific...
Finished Task about Six-Day War...
Finished Task about Mexican-American War...
Finished Task about Second Boer War...
Finished Task about Sino-Japanese War...
Expecting value: line 1 column 1 (char 0)
Finished Task about American Civil War...
Expecting value: line 1 column 1 (char 0)
Finished Task about First Italo-Ethiopian

[Document(metadata={'title': 'Second Boer War', 'summary': 'The Second Boer War was a conflict fought between the British Empire and the Boer republics (the South African Republic and Orange Free State) over Britain\'s influence in Southern Africa.\nThe Witwatersrand Gold Rush caused an influx of "foreigners" (Uitlanders), most of them British from the Cape Colony, to the South African Republic (SAR), an independent Boer Republic. As they were permitted to vote only after 14 years\' residence, they protested to the British authorities in the Cape. Negotiations failed at the botched Bloemfontein Conference in June 1899. The conflict broke out in October after the British government decided to send 10,000 troops.\nThe war had three phases. In the first, the Boers mounted preemptive strikes into British-held territory in Natal and the Cape Colony, besieging British garrisons at Ladysmith, Mafeking, and Kimberley. The Boers won victories at Stormberg, Magersfontein, Colenso and Spion Kop. 

In [ ]:
for doc in docs:
    if 'summary' in doc.metadata.keys():
        print('Removing summary...')
        doc.metadata.pop('summary')

In [66]:
docs[0].metadata

{'title': 'Second Boer War',
 'summary': 'The Second Boer War was a conflict fought between the British Empire and the Boer republics (the South African Republic and Orange Free State) over Britain\'s influence in Southern Africa.\nThe Witwatersrand Gold Rush caused an influx of "foreigners" (Uitlanders), most of them British from the Cape Colony, to the South African Republic (SAR), an independent Boer Republic. As they were permitted to vote only after 14 years\' residence, they protested to the British authorities in the Cape. Negotiations failed at the botched Bloemfontein Conference in June 1899. The conflict broke out in October after the British government decided to send 10,000 troops.\nThe war had three phases. In the first, the Boers mounted preemptive strikes into British-held territory in Natal and the Cape Colony, besieging British garrisons at Ladysmith, Mafeking, and Kimberley. The Boers won victories at Stormberg, Magersfontein, Colenso and Spion Kop. In the second phas

In [67]:
#Embedding model
embed_model = HuggingFaceEmbeddings(model= "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1047.33it/s]


In [68]:
vector_db = Chroma.from_documents(docs, embed_model)
vector_db

In [81]:
query = 'What was the contribution on India in World War I?'

In [82]:
query_embedding = embed_model.embed_query(query)

In [83]:
result = vector_db.similarity_search_by_vector_with_relevance_scores(query_embedding, 3)

In [80]:
result

[(Document(id='fb4d9856-225d-4c2e-a67d-ea4508275cb3', metadata={'title': 'War of the Pacific', 'summary': "The War of the Pacific (Spanish: Guerra del Pacífico), also known by multiple other names, was a war between Chile and a Bolivian–Peruvian alliance from 1879 to 1884. Fought over Chilean claims on coastal Bolivian territory in the Atacama Desert, the war ended with victory for Chile, which gained a significant amount of resource-rich territory from Peru and Bolivia. The war demonstrated Chile's military-technological superiority over its opponents at the time.\nThe direct cause of the war was a nitrate taxation dispute between Bolivia and Chile, with Peru being drawn in due to its secret alliance with Bolivia. Some historians have pointed to deeper origins of the war, such as the interest of Chile and Peru in the nitrate business, a long-standing rivalry between Chile and Peru for regional hegemony, as well as the political and economical disparities between the stability of Chile

In [84]:
context = ''
for doc in result:
    context+=doc[0].page_content
context

'The Second Sino-Japanese War, known in China as the War of Resistance Against Japan, was fought between the Republic of China and the Empire of Japan and its puppet states between 1937 and 1945, following a period of conflict localized to Manchuria that started in 1931. It is often regarded as the beginning of World War II in Asia, as the wars became heavily intertwined after Japan attacked the United States. It was the largest Asian war in the 20th century.\nOn 18 September 1931, the Japanese staged the Mukden incident, a false flag event fabricated to justify their invasion of Manchuria and establishment of the puppet state of Manchukuo. This is sometimes marked as the beginning of the war. From 1931 to 1937, China and Japan engaged in skirmishes, including in Shanghai and in Northern China. Nationalist and Chinese Communist Party (CCP) forces, respectively led by Chiang Kai-shek and Mao Zedong, had fought each other in the Chinese Civil War since 1927. In late 1933, Chiang Kai-shek

## Now as the vector store is prepared, get ready to feed the context to the RAG application so, it can retrieve the answer for the query.

In [85]:
resolve_query_prompt = PromptTemplate(template = 'Take help from the given text, try to answer the user query. Strictly Generate answer from within the content of the text else reply as "Context not helpful". Refer {context} to answer {query}..',
                                      input_variables=['context', 'query'])

In [86]:
query_resolve_chain = resolve_query_prompt | model_0 

In [87]:
query2 = 'Whan and where Mahatma Gandhi born and how did he die?'

In [75]:
query3 = 'Tell me about Nathhuram Godse?'

In [82]:
query4 ='Who Mahatma Gandhi died?'

In [89]:
response = query_resolve_chain.invoke({'context' : context,
                                     'query' : query })
response.pretty_print()

================================== Ai Message ==================================

Context not helpful.
